# Dataset Partitioning and Weka Export

This notebook verifies the saved cohort, maps the Any Readmission target, and exports the deterministic CSV partitions and Weka ARFF datasets.

## 1. Setup

In [ ]:
import sys
from collections import Counter
from pathlib import Path

from IPython.display import Markdown, display

working_directory = Path.cwd().resolve()
project_root = next(
    directory
    for directory in (working_directory, *working_directory.parents)
    if (directory / "src" / "paths.py").is_file() and (directory / "requirements.txt").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.preparation.data_loading import TARGET_COLUMN, TARGET_LABELS
from src.preparation.dataset_export import (
    CLASS_COLUMN,
    CLASS_LABELS,
    build_model_dataset,
    load_full_prepared,
    load_target_mapping,
    map_binary_targets,
    save_model_partitions,
    save_weka_datasets,
)

## 2. Load Full Prepared Data

The loader checks the saved CSV against its preparation manifest and the configured predictor order. The cohort table retains both identifiers and the three original target values.

In [ ]:
prepared_columns, prepared_rows = load_full_prepared()
source_counts = Counter(row[TARGET_COLUMN] for row in prepared_rows)

print(f"Prepared cohort: {len(prepared_rows):,} encounters, {len(prepared_columns)} columns")
print("Original labels:", {label: source_counts[label] for label in TARGET_LABELS})

## 3. Binary Target Mapping

Both recorded readmission categories map to the positive class. The negative class indicates that no readmission was recorded. Counts below use the full eligible cohort as the denominator and describe class prevalence before partitioning.

In [ ]:
target_mapping = load_target_mapping()
mapped_rows = map_binary_targets(prepared_rows)
model_columns, model_rows = build_model_dataset(prepared_columns, mapped_rows)
class_counts = Counter(row[CLASS_COLUMN] for row in model_rows)

table = [
    "| Original label | Model class | Encounters |",
    "| --- | --- | ---: |",
    *(f"| `{label}` | `{target_mapping[label]}` | {source_counts[label]:,} |" for label in TARGET_LABELS),
    "",
    "| Model class | Encounters | Share of cohort |",
    "| --- | ---: | ---: |",
    *(
        f"| `{label}` | {class_counts[label]:,} | {class_counts[label] / len(model_rows):.2%} |"
        for label in CLASS_LABELS
    ),
]
display(Markdown("\n".join(table)))
print(f"Internal columns: {len(prepared_columns) + 1}")
print(f"Model columns: {len(model_columns)}, with {model_columns[-1]!r} last")
del model_rows

**Observation.** Mapping preserves the cohort size. The internal records retain encounter and patient identifiers and the original labels for partitioning.

**Implication.** The model view contains the 32 ordered predictors followed by `class`.

**Limitation.** The positive class represents readmission recorded within the source data's observation coverage. The binary target combines the two source timing categories.

## 4. Stratified Partitioning and CSV Export

The split uses seed `42` with deterministic SHA-256 ranking within each original outcome group. The first stage assigns 20% of the full cohort to test. The second assigns 20% of development to validation. The remaining encounters form train.

The exporter validates patient isolation and exact cohort coverage. It saves the three partitions and a training file formed by appending validation to train. The split manifest holds the encounter and patient identifiers for reconciliation.

In [ ]:
summaries = save_model_partitions(prepared_columns, mapped_rows)
del prepared_rows, mapped_rows

table = [
    "| Dataset | Rows | Columns | Original NO | Original <30 | Original >30 | Not readmitted | Readmitted | Readmitted share |",
    "| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |",
]
for name, result in summaries.items():
    original = result["original_labels"]
    binary = result["binary_classes"]
    rows = result["rows"]
    table.append(
        f"| {name} | {rows:,} | {result['columns']} | {original['NO']:,} | "
        f"{original['<30']:,} | {original['>30']:,} | {binary['not_readmitted']:,} | "
        f"{binary['readmitted']:,} | {binary['readmitted'] / rows:.2%} |"
    )

display(Markdown("\n".join(table)))

### Empty Fields After Export

The table reports the columns with missing values and the number of rows affected. The reader preserves empty CSV fields and nominal tokens such as `None` as distinct values.

In [ ]:
empty_columns = [
    column for column in model_columns
    if any(result["empty_cells_by_column"][column] for result in summaries.values())
]
table = [
    "| Dataset | " + " | ".join([*empty_columns, "Rows with at least one empty field"]) + " |",
    "| --- | " + " | ".join(["---:"] * (len(empty_columns) + 1)) + " |",
]
for name, result in summaries.items():
    counts = [result["empty_cells_by_column"][column] for column in empty_columns]
    counts.append(result["rows_with_empty_fields"])
    table.append(f"| {name} | " + " | ".join(f"{count:,}" for count in counts) + " |")

display(Markdown("\n".join(table)))

**Observation.** The membership and round-trip checks passed before the summary was displayed. Each row belongs to exactly one primary partition, and the combined dataset follows train order followed by validation order.

**Implication.** Use train to develop experiments, validation to compare them, and the combined file for final training once the modeling configuration is selected. Test is reserved for final evaluation.

**Limitation.** Outcome stratification preserves the three source label proportions approximately. The split does not establish temporal or hospital-level generalizability.

## 5. Weka ARFF Export

The exporter reads the verified CSV files and checks their values and membership against the saved cohort. The full prepared ARFF retains the three original outcomes for exploration. The five model ARFFs use the Any Readmission class.

Development nominal domains come from train. Final training domains come from train plus validation. Both test files use the same ordered records. An unseen predictor value is recorded as `?` for the applicable schema; the source CSV value remains available for reconciliation.

In [ ]:
arff_audit = save_weka_datasets()

table = [
    "| ARFF file | Rows | Attributes | Nominal domains from | Source missing fields | Unseen nominal fields |",
    "| --- | ---: | ---: | --- | ---: | ---: |",
]
for name, result in arff_audit["datasets"].items():
    missing = sum(result["source_missing_by_column"].values())
    unseen = sum(result["unseen_nominal_to_arff_missing"].values())
    table.append(
        f"| {name}.arff | {result['rows']:,} | {result['attributes']} | {result['schema_source']} | "
        f"{missing:,} | {unseen:,} |"
    )
display(Markdown("\n".join(table)))

In [ ]:
print("Additional nominal values in the final training schema:")
print(arff_audit["domains_added_in_final_schema"])

print("\nUnseen nominal values converted to ARFF missing:")
for name, result in arff_audit["datasets"].items():
    values = result["unseen_nominal_values_by_column"]
    if values:
        print(f"{name}.arff: {values}")

**Observation.** The audit distinguishes missing source fields from values made missing by a fitted nominal domain. Literal laboratory `None` and diagnosis group `Missing` remain nominal categories.

**Implication.** Use the matching test file for the schema on which the model was trained. In Weka, set the last model attribute, `class`, as the outcome and identify `readmitted` by name.

**Limitation.** A nominal code outside the fitted domain has no distinct representation in that ARFF. The full prepared ARFF uses the original outcome and is intended for exploration.